# 8장 클린 아키텍처로 구현하는 테스트 패턴

파이썬으로 구현하는 클린 아키텍처 - 8장 클린 아키텍처로 구현하는 테스트 패턴 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정 - TodoApp 코드 import를 위한 경로 구성
# Google Colab: 깃허브에서 레포 클론 후 TODOAPP_PATH 자동 설정
# 로컬 환경: 현재 디렉토리의 TodoApp 폴더를 경로로 설정
# 반드시 첫 번째로 실행 필요 (이후 셀들이 이 경로에 의존)
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_8/TodoApp'
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

## 개요

이 장에서는 클린 아키텍처의 명시적 경계가 집중적인 유닛 테스트와 통합 테스트를 통해 포괄적인 테스트 커버리지를 어떻게 달성하는지 배운다.

이 장에서 다루는 주요 주제:
* 클린 아키텍처에서의 테스트 기초
* 테스트 가능한 구성 요소 구축. 테스트 주도 접근법
* 아키텍처 경계를 넘나드는 테스트

### 00_task_test_antipattern.py

## 테스트 안티 패턴

클린 아키텍처가 유닛 테스트를 이론에서 실천으로 변환하는 방식을 ﻿살펴보자. 간단한 테스트 목표를 고려해 보자: 새 작업이 기본적으로 중간 우선순위로 설정되는지 검증하는 것이다.

In [ ]:
# [추가] 안티패턴 예제에서 사용하는 인프라 의존 클래스들의 스텁 정의
# 실제 DB/알림 서비스 없이 안티패턴 동작을 시연하기 위한 최소 구현
# Colab/로컬 모두 동일하게 동작 - 첫 번째 셀의 경로 설정 후 실행 필요
from todo_app.domain.entities.entity import Entity
from todo_app.domain.value_objects import Priority


class Database:
    """[추가] 안티패턴 예제용 스텁 - 실제 DB 없이 동작"""
    # 작업 저장 시뮬레이션 - 항상 ID 1 반환
    def save_task(self, data: dict) -> int:
        return 1
    # 작업 조회 시뮬레이션 - 기본 우선순위 반환
    def get_task(self, task_id: int) -> dict:
        return {"priority": Priority.MEDIUM}


class NotificationService:
    """[추가] 안티패턴 예제용 스텁 - 알림 발송 시뮬레이션"""
    def __call__(self, message: str) -> None:
        pass


def create_database_connection():
    """[추가] 안티패턴 예제용 DB 연결 팩토리 스텁"""
    return Database()


def create_notification_service():
    """[추가] 안티패턴 예제용 알림 서비스 팩토리 스텁"""
    return NotificationService()

In [ ]:
# 테스트 안티패턴: 도메인 엔티티가 인프라(DB, 알림)에 직접 의존하는 구조
# 문제점: 단순한 우선순위 검증에도 DB 연결과 알림 서비스가 필요
# → 테스트 설정 복잡, 실패 원인 파악 어려움, 느린 실행 속도
# Colab/로컬 모두 동일하게 동작 (외부 패키지 불필요)
class Task(Entity):
    """안티패턴: 인프라에 직접 의존하는 도메인 엔터티"""

    def __init__(self, title: str, description: str):
        self.title = title
        self.description = description
        # 안티패턴: 생성자에서 인프라 객체를 직접 생성 (의존성 주입 미사용)
        self.db = Database()  # 데이터베이스에 직접 의존
        self.notifier = NotificationService()  # 알림 서비스에 직접 의존
        self.priority = Priority.MEDIUM
        # 생성 시 즉시 DB 저장 및 알림 전송 - 도메인 로직과 인프라 로직의 혼합
        self.id = self.db.save_task(self.as_dict())
        self.notifier(f"Task {self.id} created")

    def as_dict(self):  # [추가] 안티패턴 예제가 동작하기 위한 최소 메서드
        return {"title": self.title, "description": self.description, "priority": self.priority}


def test_new_task_priority_antipattern():
    """안티패턴 테스트: 단순한 도메인 로직과 인프라 관심사를 혼합하는 사례"""
    # 기본값 테스트를 위한 불필요하게 복잡한 설정
    db_connection = create_database_connection()
    notification_service = create_notification_service()
    # 작업 생성만으로도 DB와 알림 서비스에 접근 발생
    task = Task(title="Test task", description="Test description")
    # 단순한 우선순위 속성 확인에도 DB 쿼리 필요 - 과도한 의존성
    saved_task = task.db.get_task(task.id)
    assert saved_task["priority"] == Priority.MEDIUM


# [추가] 테스트 실행
test_new_task_priority_antipattern()
print("test_new_task_priority_antipattern 통과!")

### 01_task_test_clean.py

## 클린 테스트 패턴

도메인 엔터티를 비즈니스 규칙에 집중시키면: 테스트가 한 가지만 검증하고, 외부 의존성 없이 즉시 실행되며, 실패 시 원인이 정확히 하나뿐이다. 가장 간단한 도메인 테스트에서 시작하여 아키텍처 계층을 따라 바깥쪽으로 확장해 나간다.

In [ ]:
# 클린 테스트 패턴: 순수 도메인 엔티티로 비즈니스 규칙만 검증
# 장점: 외부 의존성 없음, 즉시 실행, 실패 원인이 정확히 하나
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리만 사용)
from dataclasses import dataclass
from uuid import UUID
from todo_app.domain.value_objects import Priority  # [추가] Priority import


# 순수 도메인 엔티티 - 인프라 의존성 없이 비즈니스 속성만 보유
@dataclass
class Task:
    """클린 아키텍처: 순수 도메인 엔터티 - DB/알림 서비스와 무관"""

    title: str
    description: str
    project_id: UUID
    priority: Priority = Priority.MEDIUM  # 기본 우선순위: 비즈니스 규칙


def test_new_task_priority():
    """클린 테스트: 도메인 로직에만 집중 - 설정 최소화, 검증 명확"""
    # 준비: 테스트에 필요한 데이터만 포함 (인프라 설정 불필요)
    task = Task(
        title="Test task",
        description="Test description",
        project_id=UUID("12345678-1234-5678-1234-567812345678"),
    )
    # 검증: 정확히 한 가지만 확인 - 기본 우선순위 값
    assert task.priority == Priority.MEDIUM


# [추가] 테스트 실행
test_new_task_priority()
print("test_new_task_priority 통과!")

### 02_test_entities.py

## 엔터티 테스트

구체적인 테스트에 들어가기 전에, 테스트 여정 전반에 걸쳐 유용하게 활용될 패턴을 정립해 보자. 빌 웨이크Bill Wake가 처음 제안한 AAA(Arrange-Act-Assert) 패턴45은 테스트를 구성하는 명확한 구조를 제시하며, 클린 아키텍처의 경계와 자연스럽게 부합한다.

In [ ]:
# 엔티티 테스트: AAA(Arrange-Act-Assert) 패턴 적용
# 클린 아키텍처의 경계와 자연스럽게 부합하는 테스트 구조
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)
from datetime import datetime, timedelta
from uuid import UUID
from todo_app.domain.entities.task import Task  # [추가] 실제 Task import (complete() 메서드 포함)


def test_task_completion_captures_completion_time():
    """작업 완료 시 완료 타임스탬프 기록 검증 - AAA 패턴 적용"""
    # 준비(Arrange): 테스트 대상 엔티티 생성
    task = Task(
        title="Test task",
        description="Test description",
        project_id=UUID("12345678-1234-5678-1234-567812345678"),
    )

    # 실행(Act): 비즈니스 동작 수행 - 작업 완료 처리
    task.complete()

    # 검증(Assert): 기대 결과 확인 - 완료 시간 기록 여부
    assert task.completed_at is not None
    assert (datetime.now() - task.completed_at) < timedelta(seconds=1)


# [추가] 테스트 실행
test_task_completion_captures_completion_time()
print("test_task_completion_captures_completion_time 통과!")

### 03_test_double.py

## 테스트 더블

테스트 더블(Mock, Stub, Fake)을 사용하여 외부 의존성을 대체하는 패턴이다.

In [ ]:
# 테스트 더블(Test Double): unittest.mock을 활용한 외부 의존성 대체
# Mock 객체로 리포지토리의 동작을 시뮬레이션하여 격리된 단위 테스트 수행
# Colab/로컬 모두 동일하게 동작 (unittest.mock은 Python 표준 라이브러리)
from unittest.mock import Mock
from uuid import UUID
from todo_app.domain.entities.task import Task  # [추가] Task import

# [추가] 테스트 더블 예제에서 사용할 샘플 작업 객체
some_task = Task(
    title="Sample task",
    description="A sample task for mock demo",
    project_id=UUID("12345678-1234-5678-1234-567812345678"),
)

# Mock 객체 생성 - 호출 기록 및 반환값 설정 가능
mock_repo = Mock()
# 원하는 응답 구성 - get() 호출 시 some_task 반환하도록 설정
mock_repo.get.return_value = some_task
# Mock 리포지토리의 get() 호출 - 실제 DB 접근 없이 some_task 반환
mock_repo.get(123)
# 상호작용 검증 - get()이 정확히 한 번 호출되었는지 확인
mock_repo.get.assert_called_once()

# Mock 객체의 호출 추적 기능 시연
# 전달된 인수 확인 - call(123)
print(mock_repo.get.call_args)
# 호출 횟수 확인 - 1
print(mock_repo.get.call_count)

### 04_test_task_complete.py

## 작업 완료 테스트

작업 완료 비즈니스 규칙을 검증하는 테스트이다.

In [ ]:
# 유스케이스 테스트: Mock 의존성을 활용한 작업 완료 시나리오 검증
# 애플리케이션 계층 테스트 - 리포지토리와 알림 서비스를 Mock으로 대체
# Colab/로컬 모두 동일하게 동작 (unittest.mock은 Python 표준 라이브러리)
from unittest.mock import Mock
from uuid import UUID
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.application.use_cases.task_use_cases import CompleteTaskUseCase  # [추가]
from todo_app.application.dtos.task_dtos import CompleteTaskRequest  # [추가]


def test_successful_task_completion():
    """유스케이스 테스트: Mock 의존성으로 작업 완료 흐름 검증"""
    # 준비(Arrange): 테스트 대상과 Mock 의존성 구성
    task = Task(
        title="Test task",
        description="Test description",
        project_id=UUID("12345678-1234-5678-1234-567812345678"),
    )
    task_repo = Mock()
    task_repo.get.return_value = task  # Mock 리포지토리: 작업 조회 시 task 반환
    notification_service = Mock()  # Mock 알림 서비스

    # 유스케이스에 Mock 의존성 주입 - 의존성 역전 원칙 적용
    use_case = CompleteTaskUseCase(
        task_repository=task_repo, notification_service=notification_service
    )
    request = CompleteTaskRequest(task_id=str(task.id))
    # 실행(Act): 유스케이스 실행
    result = use_case.execute(request)

    # 검증(Assert): 결과 및 상호작용 확인
    assert result.is_success
    task_repo.save.assert_called_once_with(task)  # 리포지토리에 저장 호출 확인
    notification_service.notify_task_completed.assert_called_once_with(task)  # 알림 발송 확인


# [추가] 테스트 실행
test_successful_task_completion()
print("test_successful_task_completion 통과!")

### 05_test_controller_converts_string_id_to_uuid.py

## 컨트롤러 UUID 변환 테스트

인터페이스 어댑터 계층으로 이동하면 테스트 초점은 외부 형식과 애플리케이션 코어 간의 적절한 변환 검증으로 변환된다. 컨트롤러와 프레젠터가 이런 변환 역할을 수행하며, 이전 계층의 유닛 테스트와 마찬가지로 이 계층 외부의 모든 요소는 모의 객체로 대체한다.

In [ ]:
# 컨트롤러 테스트: 인터페이스 어댑터 계층의 형식 변환 검증
# 외부 형식(문자열 ID)을 애플리케이션 코어 형식(UUID)으로 올바르게 변환하는지 확인
# Colab/로컬 모두 동일하게 동작 (unittest.mock은 Python 표준 라이브러리)
from uuid import UUID
from unittest.mock import Mock
from todo_app.application.common.result import Result  # [추가]
from todo_app.application.dtos.task_dtos import TaskResponse  # [추가]
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.interfaces.presenters.base import TaskPresenter  # [추가]
from todo_app.interfaces.controllers.task_controller import TaskController  # [추가]


def test_controller_converts_string_id_to_uuid():
    """컨트롤러의 문자열→UUID 변환 검증 - 인터페이스 어댑터 계층 테스트"""
    # 준비: 문자열 형태의 작업 ID (외부 클라이언트에서 전달받는 형식)
    task_id = "123e4567-e89b-12d3-a456-426614174000"
    # Mock 유스케이스: 성공 결과 반환하도록 설정
    complete_use_case = Mock()
    complete_use_case.execute.return_value = Result.success(
        TaskResponse.from_entity(
            Task(
                title="Test Task",
                description="Test Description",
                project_id=UUID("12345678-1234-5678-1234-567812345678"),
            )
        )
    )
    # Mock 프레젠터: 뷰 모델 반환하도록 설정
    presenter = Mock(spec=TaskPresenter)
    presenter.present_task.return_value = Mock()  # [추가] present_task 반환값 설정

    # 컨트롤러 생성 - 모든 유스케이스를 Mock으로 주입
    controller = TaskController(
        complete_use_case=complete_use_case,
        presenter=presenter,
        create_use_case=Mock(),   # [추가] TaskController에 필요한 나머지 필드
        get_use_case=Mock(),      # [추가]
        update_use_case=Mock(),   # [추가]
        delete_use_case=Mock(),   # [추가]
    )

    # 실행: 문자열 ID로 완료 처리 요청
    controller.handle_complete(task_id=task_id)

    # 검증: 유스케이스에 전달된 요청의 UUID 변환 확인
    complete_use_case.execute.assert_called_once()
    called_request = complete_use_case.execute.call_args[0][0]
    # [수정] CompleteTaskRequest는 task_id를 문자열로 저장하되, 유효한 UUID인지 검증
    assert UUID(called_request.task_id) == UUID(task_id)


# [추가] 테스트 실행
test_controller_converts_string_id_to_uuid()
print("test_controller_converts_string_id_to_uuid 통과!")

### 06_test_presenter_formats_completion_date.py

## 프레젠터 날짜 포맷 테스트

handle_complete를 호출할 때 컨트롤러는 다음을 수행해야 한다. * 클라이언트로부터 문자열 형태의 Task ID를 받는다.

In [ ]:
# 프레젠터 테스트: 날짜 포맷 변환 검증
# 프레젠터가 도메인 데이터를 인터페이스 요구사항에 맞게 변환하는지 확인
# 영속성/비즈니스 규칙과 무관하게 데이터 형식 검증에만 집중
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)
from datetime import datetime, timezone
from uuid import UUID
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.application.dtos.task_dtos import TaskResponse  # [추가]
from todo_app.interfaces.presenters.cli import CliTaskPresenter  # [추가]


def test_presenter_formats_completion_date():
    """프레젠터의 완료 날짜 포맷 변환 검증 - CLI 표시 형식 확인"""
    # 준비: 특정 완료 시간으로 작업 생성 (결정적 테스트를 위한 고정 시간)
    completion_time = datetime(2024, 1, 15, 14, 30, tzinfo=timezone.utc)
    task = Task(
        title="Test Task",
        description="Test Description",
        project_id=UUID("12345678-1234-5678-1234-567812345678"),
    )
    task.complete()
    task.completed_at = completion_time  # 결정적 테스트를 위해 완료 시간 재정의
    task_response = TaskResponse.from_entity(task)
    presenter = CliTaskPresenter()
    # 실행: 프레젠터를 통한 뷰 모델 생성
    view_model = presenter.present_task(task_response)
    # 검증: 기대하는 날짜 형식("2024-01-15 14:30") 포함 여부
    expected_format = "2024-01-15 14:30"
    assert view_model.completion_info is not None and expected_format in view_model.completion_info


# [추가] 테스트 실행
test_presenter_formats_completion_date()
print("test_presenter_formats_completion_date 통과!")

### 07_test_presenter_provides_complete_view_model.py

## 프레젠터 뷰 모델 테스트

테스트 흐름을 보면 클린 아키텍처의 명시적 경계가 인터페이스 어댑터 테스트를 어떻게 간단하게 만드는지 알 수 있다. 유닛 테스트에서 이미 검증한 영속성, 비즈니스 규칙과 같은 관심사와 얽히지 않고 데이터 형식 검증에만 집중한다.

In [ ]:
# 프레젠터 뷰 모델 테스트: 완전한 표시 데이터 생성 검증
# 클린 아키텍처의 명시적 경계 덕분에 데이터 형식 검증에만 집중 가능
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)
from uuid import UUID
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.domain.value_objects import Priority  # [추가]
from todo_app.application.dtos.task_dtos import TaskResponse  # [추가]
from todo_app.interfaces.presenters.cli import CliTaskPresenter  # [추가]


def test_presenter_provides_complete_view_model():
    """프레젠터의 뷰 모델 완전성 검증 - 모든 표시 필드 올바른 형식 확인"""
    # 준비: 높은 우선순위의 완료된 작업 생성
    task = Task(
        title="Important Task",
        description="Testing view model creation",
        project_id=UUID('12345678-1234-5678-1234-567812345678'),
        priority=Priority.HIGH
    )
    task.complete()  # 상태를 DONE으로 전환
    task_response = TaskResponse.from_entity(task)
    presenter = CliTaskPresenter()

    # 실행: 프레젠터를 통한 뷰 모델 변환
    view_model = presenter.present_task(task_response)

    # 검증: 각 표시 필드의 올바른 형식 확인
    assert view_model.title == "Important Task"           # 제목 유지
    assert view_model.status_display == "[DONE]"          # CLI 전용 대괄호 형식
    assert view_model.priority_display == "High"          # 사람이 읽기 쉬운 형식
    assert isinstance(view_model.completion_info, str)    # 완료 정보 문자열 존재


# [추가] 테스트 실행
test_presenter_provides_complete_view_model()
print("test_presenter_provides_complete_view_model 통과!")

### 08_test_repo_handles_project_task_relationships.py

## 리포지토리 관계 처리 테스트

유닛 테스트가 명시적 인터페이스를 통해 비즈니스 규칙과 오케스트레이션 로직을 철저히 검증하기 때문에, 통합 테스트는 매우 전략적으로 진행할 수 있다. 유닛 테스트가 모의 객체를 사용해 구성 요소의 독립적 동작을 검증했다면, 통합 테스트는 구체적 구현체들이 함께 올바르게 작동하는지 확인한다.

In [ ]:
# 통합 테스트: 리포지토리의 프로젝트-작업 관계 처리 검증
# 유닛 테스트(Mock)로는 확인 불가능한 실제 저장/조회 동작을 검증
# 구체적 구현체들이 함께 올바르게 작동하는지 확인
# Colab/로컬 모두 동일하게 동작 (tempfile로 임시 디렉토리 생성)
# Colab에서 pytest 사용 시: !pip install pytest (스텁으로도 실행 가능)
import tempfile
from pathlib import Path
from todo_app.infrastructure.persistence.file import FileTaskRepository, FileProjectRepository  # [추가]
from todo_app.domain.entities.project import Project  # [추가]
from todo_app.domain.entities.task import Task  # [추가]

# [보완] pytest.fixture / tmp_path 대신 tempfile로 대체하여 노트북/Colab 실행 가능

try:  # [수정] pytest 미설치 시에도 동작하도록 보호
    import pytest
except ImportError:
    from types import SimpleNamespace
    pytest = SimpleNamespace(fixture=lambda f: f)


@pytest.fixture
def repository(tmp_path):  # tmp_path: pytest 내장 임시 디렉토리 픽스처
    """임시 디렉토리를 사용하여 리포지토리 생성"""
    return FileTaskRepository(data_dir=tmp_path)

def test_repo_handles_project_task_relationships():
    # [수정] tmp_path 대신 tempfile.mkdtemp() 사용 (노트북/Colab 환경 호환)
    tmp_path = Path(tempfile.mkdtemp())

    # 준비: 실제 파일 기반 리포지토리 생성 및 연결
    task_repo = FileTaskRepository(tmp_path)
    project_repo = FileProjectRepository(tmp_path)
    project_repo.set_task_repository(task_repo)

    # 리포지토리를 통해 프로젝트와 작업 저장
    project = Project(name="Test Project",
                      description="Testing relationships")
    project_repo.save(project)

    task = Task(title="Test Task",
                description="Testing relationships",
                project_id=project.id)
    task_repo.save(task)
    # 실행: 프로젝트와 해당 작업들을 디스크에서 다시 로드
    loaded_project = project_repo.get(project.id)

    # 검증: 프로젝트에 연관된 작업이 올바르게 로드되었는지 확인
    assert len(loaded_project.tasks) == 1
    assert loaded_project.tasks[0].title == "Test Task"


# [추가] 테스트 실행
test_repo_handles_project_task_relationships()
print("test_repo_handles_project_task_relationships 통과!")

### 09_test_repository_automatically_creates_inbox.py

## 리포지토리 Inbox 자동 생성 테스트

이 테스트는 유닛 테스트에서 포착할 수 없었던 동작을 검증한다. * 프로젝트가 디스크에서 연관된 작업을 로드한다.

In [ ]:
# 통합 테스트: 리포지토리의 Inbox 자동 생성 및 영속성 검증
# 인스턴스 재생성 후에도 동일한 Inbox 프로젝트가 유지되는지 확인
# Colab/로컬 모두 동일하게 동작 (tempfile로 임시 디렉토리 생성)
import tempfile
from pathlib import Path
from todo_app.infrastructure.persistence.file import FileProjectRepository  # [추가]
from todo_app.domain.value_objects import ProjectType  # [추가]


def test_repository_automatically_creates_inbox():
    """Inbox 자동 생성 및 영속성 검증 - 리포지토리 인스턴스 간 일관성 확인"""
    tmp_path = Path(tempfile.mkdtemp())  # [수정] tmp_path 대신 tempfile 사용 (Colab 호환)

    # 준비: 초기 리포지토리 생성 시 Inbox 자동 생성 확인
    initial_repo = FileProjectRepository(tmp_path)
    initial_inbox = initial_repo.get_inbox()
    assert initial_inbox.name == "INBOX"                    # Inbox 이름 확인
    assert initial_inbox.project_type == ProjectType.INBOX  # Inbox 타입 확인

    # 실행: 동일한 디렉토리를 가리키는 새 리포지토리 인스턴스 생성
    new_repo = FileProjectRepository(tmp_path)

    # 검증: 새 인스턴스에서도 동일한 Inbox가 유지되는지 확인
    persisted_inbox = new_repo.get_inbox()
    assert persisted_inbox.id == initial_inbox.id           # 동일한 ID 유지
    assert persisted_inbox.project_type == ProjectType.INBOX


# [추가] 테스트 실행
test_repository_automatically_creates_inbox()
print("test_repository_automatically_creates_inbox 통과!")

### 10_def test_task_creation_with_persistence.py

## 영속성 통합 테스트

이 테스트는 모의 리포지토리를 사용한 유닛 테스트로는 확인할 수 없는 동작을 검증한다. 구체적인 리포지토리 구현체가 Inbox 자동으로 생성하고 영속적으로 유지하는지 확인한다.

In [ ]:
# 영속성 통합 테스트: 유스케이스와 실제 리포지토리의 협력 검증
# Mock이 아닌 실제 파일 리포지토리를 사용하여 저장/조회 동작 확인
# Colab/로컬 모두 동일하게 동작 (tempfile로 임시 디렉토리 생성)
import tempfile
from pathlib import Path
from uuid import UUID
from unittest.mock import Mock
from todo_app.infrastructure.persistence.file import FileTaskRepository, FileProjectRepository  # [추가]
from todo_app.application.use_cases.task_use_cases import CreateTaskUseCase  # [추가]
from todo_app.application.dtos.task_dtos import CreateTaskRequest  # [추가]


def test_task_creation_with_persistence():
    """실제 파일 리포지토리를 사용한 작업 생성 유스케이스 통합 테스트"""
    tmp_path = Path(tempfile.mkdtemp())  # [수정] tmp_path 대신 tempfile 사용 (Colab 호환)

    # 준비: 실제 파일 기반 리포지토리 구성
    task_repo = FileTaskRepository(tmp_path)
    project_repo = FileProjectRepository(tmp_path)
    project_repo.set_task_repository(task_repo)

    # 유스케이스에 실제 리포지토리 주입 (Mock 아님)
    use_case = CreateTaskUseCase(
        task_repository=task_repo,
        project_repository=project_repo,
    )

    # 실행: 작업 생성 요청
    result = use_case.execute(CreateTaskRequest(title="Test Task", description="Integration test"))

    # 검증: 작업이 실제로 저장되었는지 확인
    assert result.is_success
    created_task = task_repo.get(UUID(result.value.id))  # DB에서 다시 조회
    assert created_task.project_id == project_repo.get_inbox().id  # 기본 Inbox 프로젝트에 할당 확인


# [추가] 테스트 실행
test_task_creation_with_persistence()
print("test_task_creation_with_persistence 통과!")

### 11_test_task_creation_scenarios.py

## 작업 생성 시나리오 테스트

parametrize 데코레이터 다음에 위치한 테스트 메서드에서는 파라미터 목록의 각 항목에 대해 테스트가 한 번씩 실행된다.

In [ ]:
# 파라미터화 테스트: pytest.mark.parametrize를 활용한 다중 시나리오 검증
# 동일한 테스트 로직으로 여러 입력/기대값 조합을 반복 실행
# Colab에서 pytest 사용 시: !pip install pytest (스텁으로도 실행 가능)
import tempfile
from pathlib import Path
from uuid import UUID
try:  # [수정] pytest 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    import pytest
except ImportError:
    from types import SimpleNamespace
    _mark = SimpleNamespace(parametrize=lambda *a, **kw: (lambda f: f))
    pytest = SimpleNamespace(fixture=lambda f: f, mark=_mark)
from unittest.mock import Mock
from todo_app.domain.value_objects import ProjectType, Priority  # [추가]
from todo_app.application.repositories.task_repository import TaskRepository  # [추가]
from todo_app.infrastructure.persistence.file import FileProjectRepository  # [추가]
from todo_app.application.use_cases.task_use_cases import CreateTaskUseCase  # [추가]
from todo_app.application.dtos.task_dtos import CreateTaskRequest  # [추가]


# 파라미터화: 각 시나리오별 입력 데이터와 기대 동작 정의
@pytest.mark.parametrize(
    "request_data,expected_behavior",
    [
        # 시나리오 1: 기본 작업 생성 - INBOX 프로젝트에 자동 할당
        (
            {"title": "Test Task", "description": "Basic creation"},
            {"project_type": ProjectType.INBOX, "priority": Priority.MEDIUM}
        ),
        # 시나리오 2: 명시적 프로젝트 할당
        (
            {
                "title": "Project Task",
                "description": "With project",
                "project_id": "project-uuid"
            },
            {"project_type": ProjectType.REGULAR, "priority": Priority.MEDIUM}
        ),
        # 시나리오 3: 높은 우선순위 작업
        # ... 작업 데이터
    ],
    ids=["basic-task", "project-task", "priority-task"]  # 테스트 ID로 시나리오 식별
)
def test_task_creation_scenarios(request_data, expected_behavior):
    """파라미터화 테스트: 다양한 작업 생성 시나리오 검증"""
    # [수정] tmp_path 대신 tempfile 사용 (Colab 호환)
    tmp_path = Path(tempfile.mkdtemp())

    # 준비: Mock 리포지토리와 실제 프로젝트 리포지토리 조합
    task_repo = Mock(spec=TaskRepository)
    project_repo = FileProjectRepository(tmp_path)  # INBOX 자동 생성을 위한 실제 리포지토리

    use_case = CreateTaskUseCase(
        task_repository=task_repo,
        project_repository=project_repo
    )

    # 실행: 각 시나리오의 입력 데이터로 유스케이스 실행
    result = use_case.execute(CreateTaskRequest(**request_data))

    # 검증: 기대 동작과 실제 결과 비교
    assert result.is_success
    created_task = result.value
    if expected_behavior["project_type"] == ProjectType.INBOX:
        assert UUID(created_task.project_id) == project_repo.get_inbox().id
    assert created_task.priority == expected_behavior["priority"]


# [추가] 노트북/Colab에서는 parametrize 없이 기본 시나리오만 직접 실행
test_task_creation_scenarios(
    {"title": "Test Task", "description": "Basic creation"},
    {"project_type": ProjectType.INBOX, "priority": Priority.MEDIUM}
)
print("test_task_creation_scenarios (basic-task) 통과!")

### 12_test_controller_handles_task_creation.py

## 컨트롤러 작업 생성 처리 테스트

pytest 픽스처로 계층별 테스트 의존성을 관리하면, 클린 아키텍처의 의존성 규칙이 테스트 구조를 통해 자연스럽게 강제된다. 도메인 픽스처, 애플리케이션 픽스처, 인터페이스 픽스처가 아키텍처 경계를 존중하는 계층 구조를 형성한다.

In [ ]:
# pytest 픽스처(Fixture): 계층별 테스트 의존성 관리 패턴
# 클린 아키텍처의 의존성 규칙을 테스트 구조를 통해 자연스럽게 강제
# 도메인 → 애플리케이션 → 인터페이스 순서로 픽스처 계층 구성
# Colab에서 pytest 사용 시: !pip install pytest (스텁으로도 실행 가능)
try:  # [수정] pytest 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    import pytest
except ImportError:
    from types import SimpleNamespace
    pytest = SimpleNamespace(fixture=lambda f: f)
from uuid import UUID
from unittest.mock import Mock
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.application.repositories.task_repository import TaskRepository  # [추가]
from todo_app.application.repositories.project_repository import ProjectRepository  # [추가]
from todo_app.application.use_cases.task_use_cases import CreateTaskUseCase  # [추가]
from todo_app.application.service_ports.notifications import NotificationPort  # [추가]
from todo_app.interfaces.presenters.base import TaskPresenter  # [추가]
from todo_app.interfaces.controllers.task_controller import TaskController  # [추가]


# 루트 픽스처 (tests/conftest.py) - 모든 테스트에서 공유하는 기본 데이터
@pytest.fixture
def sample_task_data():
    """테스트를 위한 기본 작업 속성 제공"""
    return {
        "title": "Test Task",
        "description": "Sample task for testing",
        "project_id": UUID("12345678-1234-5678-1234-567812345678"),
    }


# 도메인 계층 픽스처 (tests/domain/conftest.py) - 순수 엔티티만 사용
@pytest.fixture
def domain_task(sample_task_data):
    """도메인 테스트를 위한 순수 Task 엔터티 제공"""
    return Task(**sample_task_data)


# 애플리케이션 계층 픽스처 (tests/application/conftest.py) - Mock 리포지토리 포함
@pytest.fixture
def mock_task_repository(domain_task):
    """미리 구성된 Mock 리포지토리 제공 - 도메인 픽스처에 의존"""
    repo = Mock(spec=TaskRepository)
    repo.get.return_value = domain_task
    return repo


# [추가] 노트북/Colab에서 직접 실행 가능한 버전 (pytest 픽스처 없이)
def _run_controller_test():
    """pytest 픽스처 없이 직접 실행 가능한 컨트롤러 테스트"""
    # 루트 픽스처 역할: 기본 데이터
    sample_task_data = {
        "title": "Test Task",
        "description": "Sample task for testing",
        "project_id": UUID("12345678-1234-5678-1234-567812345678"),
    }
    # 도메인 픽스처 역할: 순수 엔티티
    domain_task = Task(**sample_task_data)
    # 애플리케이션 픽스처 역할: Mock 리포지토리
    mock_task_repository = Mock(spec=TaskRepository)
    mock_task_repository.get.return_value = domain_task
    # Mock 알림 포트
    mock_notification_port = Mock(spec=NotificationPort)
    # Mock 프레젠터
    mock_presenter = Mock(spec=TaskPresenter)
    mock_presenter.present_task.return_value = Mock()

    # 인터페이스 계층: 컨트롤러에 모든 의존성 주입
    task_controller = TaskController(
        create_use_case=CreateTaskUseCase(
            task_repository=mock_task_repository,
            project_repository=Mock(spec=ProjectRepository),
        ),
        complete_use_case=Mock(),   # [추가]
        get_use_case=Mock(),        # [추가]
        update_use_case=Mock(),     # [추가]
        delete_use_case=Mock(),     # [추가]
        presenter=mock_presenter,
    )
    # 테스트 입력: JSON 형태의 작업 생성 요청
    task_request_json = {"title": "Test Task", "description": "Testing task creation", "priority": "HIGH"}

    # 실행 및 검증: 컨트롤러의 작업 생성 처리
    result = task_controller.handle_create(**task_request_json)
    assert result.is_success
    mock_task_repository.save.assert_called_once()  # 리포지토리 저장 호출 확인


_run_controller_test()
print("test_controller_handles_task_creation 통과!")

### 13_test_task_deadline_approaching.py

## 기한 임박 작업 테스트

freezegun 라이브러리에서 특정 시점을 고정할 수 있는 컨텍스트 매니저를 사용할 수 있다. freeze_time() 블록 내부의 모든 코드는 해당 시점에서 시간이 멈춘 것으로 인식하며, 블록 외부 코드는 정상적인 시간 흐름을 유지한다.

In [ ]:
# 시간 의존 테스트: freezegun을 활용한 마감일 알림 시간 경계 검증
# 시간에 의존하는 비즈니스 규칙을 결정적으로(deterministic) 테스트
# Colab에서 freezegun 사용 시: !pip install freezegun (미설치 시 미래 날짜로 대체 테스트)
from datetime import datetime, timedelta, timezone
from uuid import UUID
try:  # [수정] freezegun 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    from freezegun import freeze_time
    _FREEZEGUN_AVAILABLE = True
except ImportError:
    from contextlib import contextmanager
    @contextmanager
    def freeze_time(time_str):
        """[스텁] freezegun 미설치 시 사용되는 freeze_time 스텁"""
        yield
    _FREEZEGUN_AVAILABLE = False
from unittest.mock import Mock
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.domain.value_objects import Deadline  # [추가]
from todo_app.application.service_ports.notifications import NotificationPort  # [추가]
from todo_app.application.repositories.task_repository import TaskRepository  # [추가]
from todo_app.application.use_cases.deadline_use_cases import CheckDeadlinesUseCase  # [추가]


def test_task_deadline_approaching():
    """freezegun으로 시간 고정 후 마감일 임박 알림 동작 검증"""
    # 준비: freeze_time으로 2024-01-14 12:00에 시간 고정 후 작업 생성
    with freeze_time("2024-01-14 12:00:00"):
        task = Task(
            title="Time-sensitive task",
            description="Testing deadlines",
            project_id=UUID("12345678-1234-5678-1234-567812345678"),
            due_date=Deadline(datetime(2024, 1, 15, 12, 0, tzinfo=timezone.utc)),
        )
    notification_service = Mock(spec=NotificationPort)
    mock_task_repo = Mock(spec=TaskRepository)
    mock_task_repo.get_active_tasks.return_value = [task]  # [추가] 활성 작업 목록 설정
    # 경고 임계값: 1일 이내 마감 시 알림
    use_case = CheckDeadlinesUseCase(
        task_repository=mock_task_repo,
        notification_service=notification_service,
        warning_threshold=timedelta(days=1),
    )

    # 실행: 1시간 후(13:00)로 시간 이동 - 마감까지 23시간 남음 (임계값 이내)
    with freeze_time("2024-01-14 13:00:00"):
        result = use_case.execute()

    # 검증: 마감일 임박 알림 발송 확인
    assert result.is_success
    notification_service.notify_task_deadline_approaching.assert_called_once()


# [보완] freezegun 미설치 시(Colab 등), 미래 날짜를 사용한 간소화 테스트
def test_task_deadline_approaching_no_freezegun():
    """freezegun 없이 미래 날짜로 마감일 임박 알림 테스트 (Colab 호환)"""
    now = datetime.now(timezone.utc)
    # 12시간 후 마감 - 1일 임계값 이내이므로 알림 발생 예상
    task = Task(
        title="Time-sensitive task",
        description="Testing deadlines",
        project_id=UUID("12345678-1234-5678-1234-567812345678"),
        due_date=Deadline(now + timedelta(hours=12)),
    )
    notification_service = Mock(spec=NotificationPort)
    mock_task_repo = Mock(spec=TaskRepository)
    mock_task_repo.get_active_tasks.return_value = [task]
    use_case = CheckDeadlinesUseCase(
        task_repository=mock_task_repo,
        notification_service=notification_service,
        warning_threshold=timedelta(days=1),
    )
    result = use_case.execute()
    assert result.is_success
    notification_service.notify_task_deadline_approaching.assert_called_once()


# [추가] 테스트 실행 - 환경에 따라 적절한 버전 선택
if _FREEZEGUN_AVAILABLE:
    test_task_deadline_approaching()
    print("test_task_deadline_approaching 통과!")
else:
    test_task_deadline_approaching_no_freezegun()
    print("test_task_deadline_approaching 통과! (freezegun 없이 미래 날짜로 테스트)")